# 01 — Fingerprint-based Molecular Embeddings
Computes count-based Morgan fingerprints (radius=2, size=1024) for all molecules in an SDF file.

**Output:** CSV with shape `(N_molecules, 1024)`.

## Configuration
Set input/output paths and fingerprint parameters.

In [ ]:
INPUT_SDF  = "data/molecules.sdf"          # path to input SDF file
OUTPUT_CSV = "data/fingerprint_embeddings.csv"  # path to output CSV

RADIUS  = 2     # Morgan fingerprint radius
FP_SIZE = 1024  # fingerprint vector size

## Dependencies

In [ ]:
# pip install rdkit pandas numpy

## 1. Load molecules from SDF

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd
import numpy as np

supplier = Chem.SDMolSupplier(INPUT_SDF)
mols = list(supplier)

valid   = [(i, mol) for i, mol in enumerate(mols) if mol is not None]
invalid = [i for i, mol in enumerate(mols) if mol is None]

print(f"Total entries   : {len(mols)}")
print(f"Valid molecules : {len(valid)}")
if invalid:
    print(f"Invalid (skipped): {invalid}")

## 2. Compute Morgan fingerprints

In [ ]:
generator = AllChem.GetMorganGenerator(radius=RADIUS, fpSize=FP_SIZE, countSimulation=True)

data = []
for i, mol in valid:
    fps = generator.GetCountFingerprint(mol)
    data.append(list(fps))

df = pd.DataFrame(data, columns=[f"Bit_{j}" for j in range(FP_SIZE)])
print(f"Fingerprint matrix shape: {df.shape}")

## 3. Check bit density
Average bit density should be below 30–40% to confirm the fingerprint size is appropriate.
If density is too high, consider increasing `FP_SIZE`.

In [ ]:
bit_counts = df.iloc[:, 1:].sum(axis=1)
density    = (np.mean(bit_counts) / FP_SIZE) * 100
print(f"Average bit density: {density:.2f}%")
if density > 40:
    print("Warning: high density — consider increasing FP_SIZE.")

## 4. Save to CSV

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}  |  shape: {df.shape}")

## 5. Compute fingerprint for reference molecule

In [ ]:
REFERENCE_SDF     = "data/adagrasib.sdf"          # path to reference molecule SDF
REFERENCE_CSV_OUT = "data/adagrasib_fingerprint.csv"  # output CSV

supplier_ref = Chem.SDMolSupplier(REFERENCE_SDF)
mol_ref = [mol for mol in supplier_ref if mol is not None]
assert len(mol_ref) == 1, f"Expected 1 reference molecule, got {len(mol_ref)}"

fps_ref = generator.GetCountFingerprint(mol_ref[0])
df_ref  = pd.DataFrame([list(fps_ref)], columns=[f"Bit_{j}" for j in range(FP_SIZE)])
df_ref.to_csv(REFERENCE_CSV_OUT, index=False)
print(f"Saved: {REFERENCE_CSV_OUT}  |  shape: {df_ref.shape}")